In [5]:
import json
import re
import numpy as np
from scipy.stats import norm

def extract_action(text):
    """Извлекает действие из ответа модели с помощью простого поиска."""
    pattern = r'```(.+?)```|`(.+?)`|ˋˋˋ(.+?)ˋˋˋ'
    match = re.search(pattern, text, re.DOTALL)
    if match:
        groups = match.groups()
        for group in groups:
            if group:
                return group.strip()
    return None

def calculate_confidence_interval(accuracy, n, confidence=0.95):
    """Вычисляет доверительный интервал для accuracy."""
    z = norm.ppf(1 - (1 - confidence) / 2)
    margin_of_error = z * np.sqrt((accuracy * (1 - accuracy)) / n)
    return (accuracy - margin_of_error, accuracy + margin_of_error)

def main():
    # Загружаем данные
    with open('handcrafted_stepwise_results.json', 'r') as f:
        data = json.load(f)
    
    models = ['gpt4o', 'gemini15pro', 'gemini15flash']
    correct_predictions = {model: 0 for model in models}
    total_predictions = len(data)
    
    for item in data:
        golden_truth = item['golden_truth']
        
        for model in models:
            model_output = item.get(model, '')
            extracted_action = extract_action(model_output)
            
            if extracted_action and extracted_action == golden_truth:
                correct_predictions[model] += 1
    
    print("=== Accuracy Results ===")
    for model in models:
        accuracy = correct_predictions[model] / total_predictions
        ci_low, ci_high = calculate_confidence_interval(accuracy, total_predictions)
        
        print(f"{model}: {accuracy:.4f} ({correct_predictions[model]}/{total_predictions})")
        print(f"95% Confidence Interval: [{ci_low:.4f}, {ci_high:.4f}]")
        print()

In [6]:
main()

=== Accuracy Results ===
gpt4o: 0.4600 (23/50)
95% Confidence Interval: [0.3219, 0.5981]

gemini15pro: 0.3200 (16/50)
95% Confidence Interval: [0.1907, 0.4493]

gemini15flash: 0.3200 (16/50)
95% Confidence Interval: [0.1907, 0.4493]

